# Analyze BEAST per-frame latents (pair with keypoints)

Validate that the unsupervised BEAST latents from
`behavior_784803_2025-07-03_13-55-13/.../bottom_camera.mp4` are meaningful and
pair usefully with Lightning Pose keypoint tracking, before scaling to a
multi-session backbone.

Pipeline recap: decimate video -> `beast extract` -> `beast train` (ResNet-AE,
12 latents) -> `beast predict` -> per-frame latents `bottom_camera.npy`
shape `(1899809, 12)`.

Run top-to-bottom in a JupyterLab Cloud Workstation with the latents +
keypoint assets attached under `/data`. Linear and minimal by design.

## Setup

Imports and explicit asset paths (data assets mount read-only under `/data`).

In [ ]:
%matplotlib inline
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = "/data"
EXPECTED_FRAMES = 1_899_809
NUM_LATENTS = 12

# --- BEAST latents (captured beast-output asset) ---
LATENTS_NPY = f"{DATA}/behavior_784803_2025-07-03_13-55-13_BEASTtest/bottom_camera.npy"

# --- Raw Lightning Pose predictions (DLC-style CSV, one row per video frame) ---
LP_RAW_CSV = (
    f"{DATA}/behavior_784803_2025-07-03_13-55-13_videoprocessed_2025-11-07_21-12-28"
    "/pred_outputs/video_preds/bottom_camera_predictions.csv"
)

# --- Behavior video acquisition CSV: headerless, one row per acquired frame,
#     giving each frame's behavior_time (harp). This is the exact frame<->time
#     bridge used to align session-time event tables to latent rows. ---
VIDEO_CSV = f"{DATA}/behavior_784803_2025-07-03_13-55-13/behavior-videos/bottom_camera.csv"
BEHAV_CSV_COLUMNS = ["Behav_Time", "Frame", "Camera_Time", "Gain", "Exposure"]

# --- Processed keypoints / kinematics / events (Parquet), under intermediate_data ---
KP_DIR = (
    f"{DATA}/keypoint_tracking_bottomview_LCrecordings_20260403"
    "/behavior_784803_2025-07-03_13-55-13/intermediate_data"
)
KPS_RAW_TONGUE = f"{KP_DIR}/kps_raw_tongue_tip_center.parquet"  # per-bodypart raw (one of many)
TONGUE_KINS = f"{KP_DIR}/tongue_kins.parquet"                   # precomputed tongue kinematics
TONGUE_MOVS = f"{KP_DIR}/tongue_movs.parquet"                   # tongue movement bouts
TONGUE_TRIALS = f"{KP_DIR}/tongue_trials.parquet"              # per-trial tongue summary
NWB_LICKS = f"{KP_DIR}/nwb_df_licks.parquet"                    # lick events (from NWB)
NWB_TRIALS = f"{KP_DIR}/nwb_df_trials.parquet"                  # trial events (from NWB)


def require(path):
    """Return path if it exists, else raise with a directory listing."""
    if not os.path.exists(path):
        parent = os.path.dirname(path)
        listing = (
            "\n".join(sorted(os.listdir(parent))) if os.path.isdir(parent)
            else "(parent missing)"
        )
        raise FileNotFoundError(f"Not found: {path}\nContents of {parent}:\n{listing}")
    return path


## 1. Load latents

Find `bottom_camera.npy` (the captured beast-output asset) and assert its shape/dtype.

In [ ]:
npy_path = require(LATENTS_NPY)
print("latents:", npy_path)

z = np.load(npy_path)
print("shape:", z.shape, "dtype:", z.dtype)

assert z.ndim == 2, f"expected 2-D latents, got {z.shape}"
assert z.shape[1] == NUM_LATENTS, f"expected {NUM_LATENTS} latents, got {z.shape[1]}"
if z.shape[0] != EXPECTED_FRAMES:
    print(f"[warn] frame count {z.shape[0]} != expected {EXPECTED_FRAMES}")
z = z.astype(np.float32, copy=False)
N_FRAMES = z.shape[0]


## 2. Latent sanity check (the key gate)

Per-dim mean/std should show **real spread**, not the ~0.001 collapse seen in
the 2-epoch smoke test. Quick histograms + a short-window trace of a few dims.

In [ ]:
stats = pd.DataFrame({
    "mean": z.mean(axis=0),
    "std": z.std(axis=0),
    "min": z.min(axis=0),
    "max": z.max(axis=0),
}, index=[f"z{i}" for i in range(NUM_LATENTS)])
stats.loc["__all__"] = [z.mean(), z.std(), z.min(), z.max()]
print(stats.round(4))

collapsed = (stats["std"].iloc[:NUM_LATENTS] < 1e-2).sum()
print(f"\ndims with std < 1e-2 (near-collapse): {collapsed} / {NUM_LATENTS}")
assert collapsed < NUM_LATENTS, "ALL latent dims collapsed -- training failed."


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(14, 8), sharex=False)
for i, ax in enumerate(axes.ravel()):
    ax.hist(z[:, i], bins=80, color="steelblue")
    ax.set_title(f"z{i}  (std={z[:, i].std():.3f})", fontsize=9)
fig.suptitle("Per-dim latent distributions (all frames)")
fig.tight_layout()
plt.show()


In [ ]:
WIN = slice(0, 2000)  # short window of frames
frames = np.arange(WIN.start, WIN.stop)

# Each latent dim has a large, different baseline (means span ~[-28, 24]), which
# hides the temporal dynamics on a shared axis. Standardize each dim (global
# mean/std) and offset-stack the traces so per-dim wiggle is visible.
zs = (z[WIN] - z.mean(axis=0)) / (z.std(axis=0) + 1e-8)
SPACING = 5  # vertical gap between traces, in std units

fig, ax = plt.subplots(figsize=(14, 8))
for i in range(NUM_LATENTS):
    ax.plot(frames, zs[:, i] + i * SPACING, lw=0.8)
    ax.text(WIN.start, i * SPACING, f"z{i} ", ha="right", va="center", fontsize=9)
ax.set_xlabel("frame")
ax.set_ylabel("per-dim z-score (offset per dim)")
ax.set_yticks([])
ax.margins(x=0.02)
ax.set_title(f"Latent traces (per-dim z-scored, stacked) frames {WIN.start}:{WIN.stop}")
fig.tight_layout()
plt.show()


## 3. Load Lightning Pose keypoints

DLC/LP-style CSV under `.../intermediate_data`: multi-row header
(scorer / bodypart / coord), one row per video frame. The loader handles a
2- or 3-row header and returns per-keypoint `x, y, likelihood`.

In [ ]:
csv_path = require(LP_RAW_CSV)
print("keypoints:", csv_path)


def load_lp_csv(path):
    """Load a DLC/LP CSV. Returns (df, bodyparts). Columns are MultiIndex
    (bodypart, coord) with coord in {x, y, likelihood}."""
    for header in ([0, 1, 2], [0, 1]):
        try:
            df = pd.read_csv(path, header=header, index_col=0)
        except Exception:
            continue
        cols = df.columns
        # Drop the scorer level if present (3-row header) so we are left with
        # (bodypart, coord).
        if cols.nlevels == 3:
            df.columns = cols.droplevel(0)
        if df.columns.nlevels == 2 and {"x", "y"}.issubset(
            set(df.columns.get_level_values(1))
        ):
            bodyparts = list(dict.fromkeys(df.columns.get_level_values(0)))
            return df, bodyparts
    raise ValueError(f"Could not parse LP CSV header: {path}")


kp, BODYPARTS = load_lp_csv(csv_path)
print("rows:", len(kp), " bodyparts:", BODYPARTS)
kp.head()


## 4. Align latents <-> keypoints

Row `i` of the latents should correspond to frame `i` of the keypoints. Assert
equal length; if they differ, truncate to the common length and flag it (e.g.
if LP ran on a different frame set).

In [ ]:
n_kp = len(kp)
print(f"latent frames: {N_FRAMES}   keypoint frames: {n_kp}")

if n_kp != N_FRAMES:
    n = min(n_kp, N_FRAMES)
    print(f"[warn] length mismatch ({n_kp} vs {N_FRAMES}); truncating both to {n}.")
    z_a = z[:n]
    kp_a = kp.iloc[:n]
else:
    print("OK: keypoint row count == latent row count.")
    z_a = z
    kp_a = kp
N = z_a.shape[0]


## 5. Basic latent <-> keypoint relationship (the core test)

Per-frame keypoint speed `v = sqrt(dx^2 + dy^2)`, then compare latents against
that speed: overlaid traces, a correlation matrix, and an optional linear fit.

In [ ]:
# Pick a key bodypart; prefer a tongue/jaw keypoint if present.
PREFERRED = ["tongue_tip_center", "tongue_tip", "tongue", "jaw"]
KP_NAME = next((b for b in PREFERRED if b in BODYPARTS), BODYPARTS[0])
print("using bodypart:", KP_NAME)

x = kp_a[(KP_NAME, "x")].to_numpy(dtype=np.float32)
y = kp_a[(KP_NAME, "y")].to_numpy(dtype=np.float32)
if (KP_NAME, "likelihood") in kp_a.columns:
    lik = kp_a[(KP_NAME, "likelihood")].to_numpy(dtype=np.float32)
else:
    lik = np.ones(N, dtype=np.float32)

# Per-frame speed (pad first frame so it stays length N), mirroring kinematics.
speed = np.empty(N, dtype=np.float32)
speed[0] = 0.0
speed[1:] = np.sqrt(np.diff(x) ** 2 + np.diff(y) ** 2)
print(f"speed: mean={np.nanmean(speed):.3f} max={np.nanmax(speed):.3f} "
      f"frac_nan={np.isnan(speed).mean():.3f}")


In [ ]:
# Overlay latent dims (z-scored + offset-stacked) against keypoint speed.
W = slice(0, 2000)
wf = np.arange(W.start, W.stop)
zs2 = (z_a[W] - z_a.mean(axis=0)) / (z_a.std(axis=0) + 1e-8)
SP = 5
fig, (a0, a1) = plt.subplots(2, 1, figsize=(14, 9), sharex=True,
                             gridspec_kw={"height_ratios": [3, 1]})
for i in range(NUM_LATENTS):
    a0.plot(wf, zs2[:, i] + i * SP, lw=0.7)
    a0.text(W.start, i * SP, f"z{i} ", ha="right", va="center", fontsize=8)
a0.set_ylabel("latents (z-scored, stacked)")
a0.set_yticks([])
a0.set_title(f"Latents vs {KP_NAME} speed, frames {W.start}:{W.stop}")
a1.plot(np.arange(W.start, W.stop), speed[W], color="crimson", lw=0.9)
a1.set_ylabel(f"{KP_NAME} speed")
a1.set_xlabel("frame")
fig.tight_layout()
plt.show()


In [ ]:
# Correlation matrix: 12 latent dims vs {x, y, speed} of the chosen keypoint.
feat = pd.DataFrame({f"z{i}": z_a[:, i] for i in range(NUM_LATENTS)})
feat[f"{KP_NAME}_x"] = x
feat[f"{KP_NAME}_y"] = y
feat[f"{KP_NAME}_speed"] = speed

# Restrict to confidently-tracked frames to avoid LP dropout artefacts.
mask = np.isfinite(speed) & (lik > 0.9)
print(f"frames used (likelihood>0.9 & finite): {mask.sum()} / {N}")
corr = feat[mask].corr()

kp_cols = [f"{KP_NAME}_x", f"{KP_NAME}_y", f"{KP_NAME}_speed"]
sub = corr.loc[[f"z{i}" for i in range(NUM_LATENTS)], kp_cols]
fig, ax = plt.subplots(figsize=(5, 7))
im = ax.imshow(sub.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(kp_cols)), kp_cols, rotation=45, ha="right")
ax.set_yticks(range(NUM_LATENTS), [f"z{i}" for i in range(NUM_LATENTS)])
for (r, c), val in np.ndenumerate(sub.values):
    ax.text(c, r, f"{val:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, label="Pearson r")
ax.set_title("Latent <-> keypoint correlation")
fig.tight_layout()
plt.show()

print("\nmax |corr| with speed:", sub[f"{KP_NAME}_speed"].abs().max().round(3))


In [ ]:
# (Optional) linear fit: latents -> keypoint x and y; report R^2.
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

Xz = z_a[mask]
for target, name in [(x[mask], "x"), (y[mask], "y"), (speed[mask], "speed")]:
    reg = LinearRegression().fit(Xz, target)
    r2 = r2_score(target, reg.predict(Xz))
    print(f"latents -> {KP_NAME}_{name}:  R^2 = {r2:.3f}")


## 6. (Optional, light) Latent structure

PCA variance of the 12 dims, and a 2-D scatter of a ~20k-frame subsample
colored by keypoint speed. UMAP skipped this pass to avoid a new dependency.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA().fit(z_a)
evr = pca.explained_variance_ratio_
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(1, NUM_LATENTS + 1), evr, color="slateblue")
ax.plot(range(1, NUM_LATENTS + 1), np.cumsum(evr), "-o", color="black",
        label="cumulative")
ax.set_xlabel("PC")
ax.set_ylabel("explained variance ratio")
ax.set_title("PCA of the 12 latent dims")
ax.legend()
fig.tight_layout()
plt.show()
print("cumulative EVR:", np.cumsum(evr).round(3))


In [ ]:
# 2-D PCA scatter of a subsample, colored by keypoint speed.
rng = np.random.default_rng(0)
idx = np.where(mask)[0]
sub_idx = rng.choice(idx, size=min(20_000, idx.size), replace=False)
proj = pca.transform(z_a[sub_idx])[:, :2]
c = np.clip(speed[sub_idx], 0, np.nanpercentile(speed[sub_idx], 99))

fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(proj[:, 0], proj[:, 1], c=c, s=4, cmap="viridis", alpha=0.6)
fig.colorbar(sc, ax=ax, label=f"{KP_NAME} speed")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title(f"Latent PCA subsample (n={sub_idx.size}) colored by speed")
fig.tight_layout()
plt.show()


## 7. Processed kinematics & events (precomputed assets)

Richer per-bodypart/kinematics/event tables live under `intermediate_data` as
Parquet. Schemas are confirmed at runtime (printed below), so these cells just
load and peek — they're the hooks for follow-up event-aligned analysis (e.g.
latent trajectories around lick onset) once columns are known.

In [ ]:
def peek_parquet(path, n=3):
    df = pd.read_parquet(require(path))
    print(f"{os.path.basename(path)}: shape={df.shape}")
    print("columns:", list(df.columns))
    print("index:", df.index.name, type(df.index).__name__)
    return df


# Precomputed tongue kinematics (e.g. speed/displacement per frame).
tongue_kins = peek_parquet(TONGUE_KINS)
tongue_kins.head()


In [ ]:
# Per-bodypart raw keypoints (one file per label; tongue_tip_center shown).
kps_raw = peek_parquet(KPS_RAW_TONGUE)
kps_raw.head()


In [ ]:
# Behavioral events from NWB: licks and trials. Movement bouts / per-trial
# tongue summaries are also available (TONGUE_MOVS / TONGUE_TRIALS).
licks = peek_parquet(NWB_LICKS)
trials = peek_parquet(NWB_TRIALS)
licks.head()


## 8. Dynamics & event-aligned analysis

BEAST encodes *static* per-frame pose, so the latents linearly decode tongue
**position** well but instantaneous **speed** poorly. Movement lives in how the
latents *change*, so here we (a) relate latent velocity to keypoint speed,
(b) denoise by smoothing before differencing, and (c) align latents to
**tongue-movement onsets** (`tongue_movs`).

**Time-base alignment.** Latents are indexed by *video frame*; `tongue_movs`
times are in *session_time* (first go cue = 0). Three clocks (per
`aind-dynamic-foraging-behavior-video-analysis`, branch `video_alignment`):

| clock | zero |
|---|---|
| `behavior_time` | absolute harp time (video CSV `Behav_Time`, NWB `goCue_start_time_raw`) |
| `session_time` | first go cue = 0 (`tongue_movs.start_time`) |
| frame | latent row i = `bottom_camera.csv` row i |

Map a session-time event to a latent frame exactly (no fps assumption, robust to
dropped frames): `behavior_time = start_time + goCue_start_time_raw[0]`, then
`frame = searchsorted(Behav_Time_per_frame, behavior_time)`.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# (a) Latent VELOCITY (per-dim |frame-to-frame change|) -> keypoint speed.
dz = np.zeros_like(z_a)
dz[1:] = np.abs(np.diff(z_a, axis=0))
m = mask & np.r_[False, mask[:-1]]          # valid on both frame i and i-1
reg = LinearRegression().fit(dz[m], speed[m])
print("latent-velocity -> tongue speed:  R^2 =",
      round(r2_score(speed[m], reg.predict(dz[m])), 3))
print("||dz|| vs tongue speed:  r =",
      round(np.corrcoef(np.linalg.norm(dz, axis=1)[m], speed[m])[0, 1], 3))

# (b) Smooth latents before differencing to cut per-frame AE jitter, sweep K.
for K in (3, 5, 9):
    z_sm = pd.DataFrame(z_a).rolling(K, center=True, min_periods=1).mean().to_numpy()
    dz_sm = np.zeros_like(z_sm)
    dz_sm[1:] = np.abs(np.diff(z_sm, axis=0))
    r = LinearRegression().fit(dz_sm[m], speed[m])
    print(f"smoothed(K={K}) latent-velocity -> speed:  R^2 =",
          round(r2_score(speed[m], r.predict(dz_sm[m])), 3))


In [ ]:
# (c) Map tongue-movement onsets (session_time) -> exact latent frames using
# the per-frame behavior_time from the video CSV (see video_alignment module).
behav = pd.read_csv(require(VIDEO_CSV), names=BEHAV_CSV_COLUMNS)
bt = behav["Behav_Time"].to_numpy()          # per-frame behavior_time (harp), sorted
assert np.all(np.diff(bt) >= 0), "Behav_Time not monotonic; cannot searchsorted"
print(f"video CSV frames: {len(bt)}  (latents N={N})")
if len(bt) != N:
    print(f"[warn] CSV frame count {len(bt)} != latent rows {N}; "
          "row i<->frame i alignment may be off.")

trials = pd.read_parquet(require(NWB_TRIALS))
first_go_cue = float(trials["goCue_start_time_raw"].iloc[0])   # behavior_time of 1st go cue
print(f"first frame behavior_time={bt[0]:.3f}s  first go cue={first_go_cue:.3f}s  "
      f"video_time of go cue={first_go_cue - bt[0]:.3f}s")

movs = pd.read_parquet(require(TONGUE_MOVS))
onset_bt = movs["start_time"].to_numpy() + first_go_cue        # session_time -> behavior_time
onset = np.searchsorted(bt, onset_bt)                          # behavior_time -> frame/row
onset = onset[(onset >= 0) & (onset < N)]

med_len = int(np.median((movs["n_datapoints"] + movs.get("dropped_frames_n", 0)).clip(lower=1)))
HALF = int(np.clip(3 * med_len, 30, 1000))    # window scales with movement length
print(f"{len(onset)} onsets mapped (frames {onset.min()}..{onset.max()}, N={N}); "
      f"median movement = {med_len} frames; using +/-{HALF}-frame window")


In [ ]:
# Peri-onset averages: keypoint speed, latent speed, and per-dim z-scored latents.
win = np.arange(-HALF, HALF + 1)
valid = onset[(onset - HALF >= 0) & (onset + HALF < N)]
idx = valid[:, None] + win[None, :]          # (n_events, n_win)

z_sm = pd.DataFrame(z_a).rolling(5, center=True, min_periods=1).mean().to_numpy()
lat_speed = np.zeros(N)
lat_speed[1:] = np.linalg.norm(np.diff(z_sm, axis=0), axis=1)
zz = (z_a - z_a.mean(0)) / (z_a.std(0) + 1e-8)

peri_kp = speed[idx]
peri_lat = lat_speed[idx]
peri_z = np.nanmean(zz[idx], axis=0)          # (n_win, 12)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(win, np.nanmean(peri_kp, axis=0), color="crimson", label=f"{KP_NAME} speed")
ax[0].set_title(f"Keypoint speed around onset (n={len(valid)})")
ax[1].plot(win, np.nanmean(peri_lat, axis=0), color="navy", label="||dz||")
ax[1].set_title("Latent speed ||dz|| around onset")
for a in ax:
    a.axvline(0, color="k", lw=0.8, ls="--")
    a.set_xlabel("frame from movement onset")
fig.tight_layout()
plt.show()

v = float(np.nanpercentile(np.abs(peri_z), 95)) or 0.5
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(peri_z.T, aspect="auto", cmap="RdBu_r", vmin=-v, vmax=v,
               extent=[win[0], win[-1], NUM_LATENTS - 0.5, -0.5])
ax.set_yticks(range(NUM_LATENTS), [f"z{i}" for i in range(NUM_LATENTS)])
ax.axvline(0, color="k", lw=0.8, ls="--")
ax.set_xlabel("frame from movement onset")
ax.set_title("Mean z-scored latents around tongue-movement onset")
fig.colorbar(im, ax=ax, label="z")
fig.tight_layout()
plt.show()


## Summary / gates

- **Latent gate:** per-dim std clearly non-zero / varied (latents not collapsed);
  PCA shows the 12 dims carry structure.
- **Keypoint gate:** frame alignment succeeded and at least one latent<->keypoint
  relationship is quantified (correlation and regression R^2 above).

If both gates pass, the latents are meaningful enough to proceed toward the
multi-session backbone (see PLAN.md roadmap).